In [22]:
import cv2
import numpy as np
import os
from collections import Counter

class BuscadorDeImagens:
    def __init__(self, color_space='gray', bins=256):
        self.color_space = color_space
        self.bins = bins

    def extrair_pdf(self, image_path,aplicar_quantizacao_imagem=False,aplicar_filtro=False):
        img = cv2.imread(image_path)
        if img is None:
            raise ValueError(f"Não foi possível carregar a imagem: {image_path}")
        # Se quiser alterar fisicamente os pixels da imagem agrupando os valores
        if aplicar_filtro:
            # Aplica um desfoque leve (kernel 5x5) para remover ruídos pontuais
            img = cv2.GaussianBlur(img, (5, 5), 0)
        if aplicar_quantizacao_imagem:
            # Fator de redução. Ex: 256 / 64 = 4.
            # Agrupa os valores de 4 em 4 usando divisão inteira
            fator = 256 // self.bins
            img = (img // fator) * fator
        if self.color_space == 'gray':
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            hist = cv2.calcHist([img], [0], None, [self.bins], [0, 256])
        elif self.color_space == 'rgb':
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            hist = cv2.calcHist([img], [0, 1, 2], None, [self.bins]*3, [0, 256]*3)
        elif self.color_space == 'hsv':
            img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
            # Foi ignorado a iluminacao
            hist = cv2.calcHist([img], [0, 1], None, [self.bins, self.bins], [0, 180, 0, 256])

        # Normalização do histograma (Soma = 1) -> Transforma em PDF
        cv2.normalize(hist, hist, alpha=1, norm_type=cv2.NORM_L1)
        return hist.flatten()

    def calcular_distancia(self, pdf1, pdf2, metrica='euclidiana'):
        if metrica == 'euclidiana':
            return np.linalg.norm(pdf1 - pdf2)
        # Outras métricas podem ser adicionadas aqui (ex: Chi-Quadrado)

    def buscar(self, query_path, db_folder, N=2):
        query_pdf = self.extrair_pdf(query_path)
        resultados = []

        for filename in os.listdir(db_folder):
            if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue

            img_path = os.path.join(db_folder, filename)
            # Caso a imagem esteja presente no proprio dataset ela é eliminada do rank
            if os.path.abspath(query_path) == os.path.abspath(img_path):
                continue

            db_pdf = self.extrair_pdf(img_path)
            distancia = self.calcular_distancia(query_pdf, db_pdf)
            resultados.append((distancia, filename))

        resultados.sort(key=lambda x: x[0])
        return resultados[:N]

    def extrair_classe_do_nome(self, filename):

        return filename.split('_')[0].lower()

    def avaliar_taxa_acerto(self, db_folder, N=3):
        imagens = [f for f in os.listdir(db_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        acertos = 0
        total_consultas = len(imagens)

        for query_filename in imagens:
            query_path = os.path.join(db_folder, query_filename)
            classe_real = self.extrair_classe_do_nome(query_filename)

            resultados = self.buscar(query_path, db_folder, N)

            classes_recuperadas = [self.extrair_classe_do_nome(res[1]) for res in resultados]

            classe_predita = Counter(classes_recuperadas).most_common(1)[0][0]

            if classe_predita == classe_real:
                acertos += 1

        taxa_acerto = acertos / total_consultas
        return taxa_acerto


In [23]:
pasta_banco = 'Vistex'
nome_arquivo_log = "resultados_vistex_log.txt"
from datetime import datetime
import os
for classe_n in range(1, 54):

    # Pega a data e hora atuais para registrar no log
    agora = datetime.now().strftime("%d/%m/%Y %H:%M:%S")

    # Abre o arquivo em modo 'a' (append) para adicionar os logs sem apagar os antigos
    with open(nome_arquivo_log, "a", encoding="utf-8") as arquivo_log:

        # Função auxiliar para imprimir no terminal E salvar no .txt simultaneamente
        def print_log(mensagem):
            print(mensagem)
            arquivo_log.write(mensagem + "\n")

        print_log(f"\n{'='*60}")
        print_log(f"INÍCIO DOS TESTES CBIR - DATA E HORA: {agora}")
        print_log(f"{'='*60}")

        for classe_n in range(1, 54):
            # CORREÇÃO: Garante o formato de 3 dígitos (ex: c001, c010, c053)
            imagem_consulta = f'Vistex/c{classe_n:03d}_001.png'

            for metodo_busca in ["rgb", "gray", "hsv"]:
                for aplicar_filtro in [False,True,False,True]:
                    for aplicar_quantizacao_imagem in [False,False,True,True]:
                        print_log(f"\n>>> Testando classe: {classe_n:02d} | Espaço de Cor: {metodo_busca.upper()} | filtro: {aplicar_filtro} | quantizacao: {aplicar_quantizacao_imagem} |")

                        buscador = BuscadorDeImagens(color_space=metodo_busca, bins=32)
                        try:
                            print_log("--- RECUPERANDO IMAGENS ---")
                            print_log(f"Imagem de busca: {imagem_consulta}")

                            # Recupera as 3 imagens mais parecidas
                            resultados = buscador.buscar(query_path=imagem_consulta, db_folder=pasta_banco, N=3)

                            for posicao, (distancia, nome_arquivo) in enumerate(resultados, 1):
                                print_log(f"{posicao}º lugar: {nome_arquivo} (Distância: {distancia:.4f})")

                            print_log("\n--- AVALIANDO SISTEMA ---")
                            print_log("Calculando taxa de acerto para toda a base... aguarde.")


                            taxa = buscador.avaliar_taxa_acerto(db_folder=pasta_banco, N=3)
                            print_log(f"Taxa de Acerto Global ({metodo_busca.upper()}): {taxa * 100:.2f}%")

                        except Exception as e:
                            print_log(f"Ocorreu um erro: {e}")
                            print_log("Dica: Verifique se o caminho da imagem está correto!")

            print_log(f"\n{'='*60}")
            print_log("FIM DA EXECUÇÃO")
            print_log(f"{'='*60}\n")


INÍCIO DOS TESTES CBIR - DATA E HORA: 20/03/2026 19:36:27

>>> Testando classe: 01 | Espaço de Cor: RGB | filtro: False | quantizacao: False |
--- RECUPERANDO IMAGENS ---
Imagem de busca: Vistex/c001_001.png
1º lugar: c001_014.png (Distância: 0.0369)
2º lugar: c001_005.png (Distância: 0.0387)
3º lugar: c001_010.png (Distância: 0.0388)

--- AVALIANDO SISTEMA ---
Calculando taxa de acerto para toda a base... aguarde.


KeyboardInterrupt: 